This file is for finding an appropraite loss func
Till now i was using cross entropy
I will experiment like this CrossEntropy, LabelSmoothing, Weighted CrossEntropy, Focal Loss, Focal+classWeights, Balanced Softmax

In [1]:
import pandas as pd
import regex as re
import torch

In [2]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [3]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return class_id

In [4]:
from pathlib import Path

In [5]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [6]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [7]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
from torch.utils.data import Dataset
from PIL import Image

In [9]:
class dset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = label_function(Path(image_path))
        if self.transform:
            image = self.transform(image)
        return image, label

In [10]:
image_paths = list(path.rglob("*.png"))

In [11]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = dset(image_paths = image_paths, transform = transform)

In [12]:
from torch.utils.data import random_split

ts = int(0.75*len(dataset))
vs = len(dataset) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset, [ts, vs], generator)

In [13]:
from torch.utils.data import DataLoader

In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

In [15]:
import torchvision
num_classes = 55
device = torch.device("cuda")

In [16]:
import kornia.augmentation as K
import torch.nn as nn

In [17]:
model_1 = torchvision.models.resnet34(weights="DEFAULT")
model_1.fc = nn.Linear(
    model_1.fc.in_features,
    num_classes
)

model_1 = model_1.to(device)

for param in model_1.parameters():
    param.requires_grad = False

for param in model_1.fc.parameters():
    param.requires_grad = True

train_aug_1 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_1 = torch.optim.RMSprop(
    model_1.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_1,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_1.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_1(images)

        optimizer_1.zero_grad()

        outputs = model_1(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_1.step()

        scheduler_1.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_1.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_1(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 63.70% | Valid Acc: 88.44%
Epoch 2/20 | Train Acc: 88.37% | Valid Acc: 93.06%
Epoch 3/20 | Train Acc: 85.96% | Valid Acc: 87.48%
Epoch 4/20 | Train Acc: 86.77% | Valid Acc: 87.86%
Epoch 5/20 | Train Acc: 88.40% | Valid Acc: 94.32%
Epoch 6/20 | Train Acc: 89.46% | Valid Acc: 90.46%
Epoch 7/20 | Train Acc: 88.98% | Valid Acc: 92.97%
Epoch 8/20 | Train Acc: 89.72% | Valid Acc: 94.89%
Epoch 9/20 | Train Acc: 91.49% | Valid Acc: 89.88%
Epoch 10/20 | Train Acc: 92.16% | Valid Acc: 93.64%
Epoch 11/20 | Train Acc: 90.91% | Valid Acc: 96.05%
Epoch 12/20 | Train Acc: 93.03% | Valid Acc: 94.89%
Epoch 13/20 | Train Acc: 92.68% | Valid Acc: 96.34%
Epoch 14/20 | Train Acc: 94.25% | Valid Acc: 94.22%
Epoch 15/20 | Train Acc: 95.09% | Valid Acc: 95.18%
Epoch 16/20 | Train Acc: 95.86% | Valid Acc: 97.21%
Epoch 17/20 | Train Acc: 97.85% | Valid Acc: 97.88%
Epoch 18/20 | Train Acc: 98.55% | Valid Acc: 98.36%
Epoch 19/20 | Train Acc: 99.20% | Valid Acc: 98.94%
Epoch 20/20 | Train A

In [18]:
model_2 = torchvision.models.resnet34(weights="DEFAULT")
model_2.fc = nn.Linear(
    model_2.fc.in_features,
    num_classes
)

model_2 = model_2.to(device)

for param in model_2.parameters():
    param.requires_grad = False

for param in model_2.fc.parameters():
    param.requires_grad = True

train_aug_2 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_2 = torch.optim.RMSprop(
    model_2.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_2,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss(label_smoothing = 0.1)

for epoch in range(20):
    model_2.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_2(images)

        optimizer_2.zero_grad()

        outputs = model_2(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_2.step()

        scheduler_2.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_2.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_2(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 65.47% | Valid Acc: 90.08%
Epoch 2/20 | Train Acc: 89.14% | Valid Acc: 91.33%
Epoch 3/20 | Train Acc: 88.08% | Valid Acc: 86.61%
Epoch 4/20 | Train Acc: 85.61% | Valid Acc: 82.66%
Epoch 5/20 | Train Acc: 85.83% | Valid Acc: 86.13%
Epoch 6/20 | Train Acc: 86.51% | Valid Acc: 93.74%
Epoch 7/20 | Train Acc: 85.99% | Valid Acc: 89.79%
Epoch 8/20 | Train Acc: 86.44% | Valid Acc: 83.24%
Epoch 9/20 | Train Acc: 85.74% | Valid Acc: 93.26%
Epoch 10/20 | Train Acc: 87.86% | Valid Acc: 91.04%
Epoch 11/20 | Train Acc: 88.69% | Valid Acc: 93.16%
Epoch 12/20 | Train Acc: 90.11% | Valid Acc: 90.85%
Epoch 13/20 | Train Acc: 90.68% | Valid Acc: 93.74%
Epoch 14/20 | Train Acc: 92.77% | Valid Acc: 94.99%
Epoch 15/20 | Train Acc: 93.03% | Valid Acc: 95.47%
Epoch 16/20 | Train Acc: 96.24% | Valid Acc: 97.40%
Epoch 17/20 | Train Acc: 98.52% | Valid Acc: 97.78%
Epoch 18/20 | Train Acc: 99.00% | Valid Acc: 97.98%
Epoch 19/20 | Train Acc: 99.74% | Valid Acc: 98.65%
Epoch 20/20 | Train A

In [19]:
len(image_paths)

4151

In [20]:
type(path.rglob("*.png"))

generator

In [26]:
l = list()
m = int()
for x in range(55):
    n = len(list(Path(path/f'{x}').rglob("*.png")))
    l.append(1/float(n))
    m = max(m, n)

l = [i*m for i in l]

In [27]:
l

[3.7796610169491527,
 21.238095238095237,
 5.575,
 1.664179104477612,
 4.372549019607843,
 2.2989690721649483,
 5.717948717948718,
 2.9342105263157894,
 55.75,
 223.0,
 6.371428571428571,
 3.2318840579710146,
 4.645833333333333,
 12.388888888888888,
 3.484375,
 20.272727272727273,
 3.1408450704225355,
 3.430769230769231,
 24.777777777777775,
 37.166666666666664,
 24.777777777777775,
 31.857142857142854,
 4.46,
 223.0,
 3.5396825396825395,
 15.928571428571427,
 1.0,
 10.136363636363637,
 2.3473684210526313,
 10.619047619047619,
 31.857142857142854,
 111.5,
 17.153846153846153,
 2.858974358974359,
 7.689655172413793,
 14.866666666666667,
 13.117647058823529,
 13.9375,
 24.777777777777775,
 13.9375,
 5.439024390243903,
 14.866666666666667,
 18.583333333333332,
 24.777777777777775,
 37.166666666666664,
 44.6,
 10.619047619047619,
 7.9642857142857135,
 55.75,
 12.388888888888888,
 223.0,
 1.376543209876543,
 2.753086419753086,
 4.054545454545455,
 74.33333333333333]

In [31]:
l = torch.tensor(l, dtype = torch.float32)

C:\Users\Nilansh Barotia\AppData\Local\Temp\ipykernel_6580\2052221944.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  l = torch.tensor(l, dtype = torch.float32)


In [32]:
type(l)

torch.Tensor

In [33]:
l = l.to(device)

In [ ]:
model_3 = torchvision.models.resnet34(weights="DEFAULT")
model_3.fc = nn.Linear(
    model_3.fc.in_features,
    num_classes
)

model_3 = model_3.to(device)

for param in model_3.parameters():
    param.requires_grad = False

for param in model_3.fc.parameters():
    param.requires_grad = True

train_aug_3 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_3 = torch.optim.RMSprop(
    model_3.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_3 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_3,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss(weight = l)

for epoch in range(20):
    model_3.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_3(images)

        optimizer_3.zero_grad()

        outputs = model_3(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_3.step()

        scheduler_3.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_3.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_3(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 60.46% | Valid Acc: 88.25%
Epoch 2/20 | Train Acc: 84.93% | Valid Acc: 80.15%
Epoch 3/20 | Train Acc: 83.81% | Valid Acc: 90.85%
Epoch 4/20 | Train Acc: 83.42% | Valid Acc: 84.49%
Epoch 5/20 | Train Acc: 83.46% | Valid Acc: 84.87%
Epoch 6/20 | Train Acc: 86.09% | Valid Acc: 94.03%
Epoch 7/20 | Train Acc: 87.28% | Valid Acc: 93.55%
Epoch 8/20 | Train Acc: 88.15% | Valid Acc: 86.71%
Epoch 9/20 | Train Acc: 88.89% | Valid Acc: 93.45%
Epoch 10/20 | Train Acc: 90.11% | Valid Acc: 93.64%
Epoch 11/20 | Train Acc: 88.44% | Valid Acc: 94.22%
Epoch 12/20 | Train Acc: 90.78% | Valid Acc: 94.70%
Epoch 13/20 | Train Acc: 90.81% | Valid Acc: 94.03%
Epoch 14/20 | Train Acc: 94.09% | Valid Acc: 97.11%
Epoch 15/20 | Train Acc: 93.90% | Valid Acc: 94.99%
Epoch 16/20 | Train Acc: 95.50% | Valid Acc: 94.89%
Epoch 17/20 | Train Acc: 97.33% | Valid Acc: 97.88%
Epoch 18/20 | Train Acc: 98.49% | Valid Acc: 98.27%
Epoch 19/20 | Train Acc: 99.42% | Valid Acc: 98.46%
Epoch 20/20 | Train A

In [35]:
model_4 = torchvision.models.resnet34(weights="DEFAULT")
model_4.fc = nn.Linear(
    model_4.fc.in_features,
    num_classes
)

model_4 = model_4.to(device)

for param in model_4.parameters():
    param.requires_grad = False

for param in model_4.fc.parameters():
    param.requires_grad = True

train_aug_4 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_4 = torch.optim.RMSprop(
    model_4.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_4 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_4,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss(reduction = "none")

for epoch in range(20):
    model_4.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_4(images)

        optimizer_4.zero_grad()

        outputs = model_4(images)

        loss = criterion(outputs, labels)

        focal_loss = ((1-torch.exp((-1)*loss)).pow(2)*loss).mean()

        focal_loss.backward()

        optimizer_4.step()

        scheduler_4.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_4.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_4(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 64.82% | Valid Acc: 83.62%
Epoch 2/20 | Train Acc: 86.32% | Valid Acc: 84.10%
Epoch 3/20 | Train Acc: 82.27% | Valid Acc: 88.82%
Epoch 4/20 | Train Acc: 83.49% | Valid Acc: 83.62%
Epoch 5/20 | Train Acc: 84.58% | Valid Acc: 88.25%
Epoch 6/20 | Train Acc: 88.56% | Valid Acc: 89.21%
Epoch 7/20 | Train Acc: 88.37% | Valid Acc: 91.14%
Epoch 8/20 | Train Acc: 87.79% | Valid Acc: 91.52%
Epoch 9/20 | Train Acc: 90.11% | Valid Acc: 91.23%
Epoch 10/20 | Train Acc: 89.05% | Valid Acc: 91.62%
Epoch 11/20 | Train Acc: 90.30% | Valid Acc: 91.71%
Epoch 12/20 | Train Acc: 90.52% | Valid Acc: 94.22%
Epoch 13/20 | Train Acc: 91.94% | Valid Acc: 95.47%
Epoch 14/20 | Train Acc: 93.45% | Valid Acc: 94.12%
Epoch 15/20 | Train Acc: 93.41% | Valid Acc: 96.63%
Epoch 16/20 | Train Acc: 94.80% | Valid Acc: 95.38%
Epoch 17/20 | Train Acc: 96.47% | Valid Acc: 96.92%
Epoch 18/20 | Train Acc: 98.23% | Valid Acc: 98.46%
Epoch 19/20 | Train Acc: 98.81% | Valid Acc: 98.75%
Epoch 20/20 | Train A